<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Create a Wide-Area Ethernet (Layer 2) Network: Manual Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** This notebook shows how to create an isolated **wide-area Layer 2 Ethernet** circuit connecting two nodes on **different** FABRIC sites, then **manually** assign IP addresses after the slice becomes active. A WAN L2 network creates a single Ethernet broadcast domain spanning two sites, giving you full control over addressing and protocol configuration.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create a WAN L2 network by placing interfaces on **different sites** without pre-configuring IPs
2. Inspect the provisioned slice (nodes, networks, interfaces)
3. Choose your own subnet and manually assign IP addresses with `iface.ip_addr_add()`
4. Verify cross-site L2 connectivity

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be comfortable creating basic slices (see [Hello, FABRIC](../../hello_fabric/hello_fabric.ipynb))
3. Understand local L2 networks (see [L2 Basic Manual](../create_l2network_basic/create_l2network_basic_manual.ipynb))

**Tip -- Auto vs. Manual:**
- **Auto** ([auto notebook](./create_l2network_wide_area_auto.ipynb)): You specify a subnet, FABlib assigns IPs automatically
- **Manual** (this notebook): You assign IPs yourself after the slice is active -- full control

</div>

## Background: Wide-Area L2 with Manual Configuration

With manual configuration, you submit the slice with **no subnet or IP configuration**. FABRIC creates the WAN L2 circuit and connects the nodes' NICs to it. After the slice is active, you:

1. **Choose a subnet** (any private range, e.g., `192.168.1.0/24`)
2. **Assign IPs** to each interface
3. **(Optionally)** deploy non-IP protocols -- the L2 circuit carries raw Ethernet frames


### When to Use WAN L2 Manual Configuration
- You need **specific IP addresses** for reproducibility
- You want to deploy **non-IP protocols** (custom L3, bridging experiments)
- You need **direct L2 access** between sites without routing overhead
- You want to understand the networking primitives

### How FABRIC Decides: Local vs. WAN
- If all interfaces are on the **same site** --> local L2 (virtual switch)
- If interfaces are on **different sites** --> WAN L2 (dedicated backbone circuit)

### NIC Component Models

| Model | Speed | Type | Ports |
|-------|-------|------|-------|
| `NIC_Basic` | 100 Gbps | Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps | Dedicated Mellanox ConnectX-5 | 2 |
| `NIC_ConnectX_6` | 100 Gbps | Dedicated Mellanox ConnectX-6 | 2 |

## What We're Building

In this notebook we will create two nodes on different sites connected by a wide-area L2 Ethernet link (L2STS).

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

We select two **different** random sites to create a WAN L2 circuit.

In [ ]:
# Name for the slice
slice_name = 'MySlice'

# Pick two different random FABRIC sites
[site1,site2]  = fablib.get_random_sites(count=2)
print(f"Sites: {site1}, {site2}")

# Node, network, and NIC names
node1_name = 'Node1'
node2_name = 'Node2'
network_name='net1'
node1_nic_name = 'nic1'
node2_nic_name = 'nic2'

## Step 3: Create and Submit the Slice

With manual configuration, we create the nodes with NICs and connect them to an L2 network **without** specifying a subnet. Since the interfaces are on different sites, FABRIC creates a WAN L2 circuit.

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# --- Node1 on site1 with a NIC ---
node1 = slice.add_node(name=node1_name, site=site1)
# Add a NIC_Basic component and get its first (only) interface
iface1 = node1.add_component(model='NIC_Basic', name=node1_nic_name).get_interfaces()[0]

# --- Node2 on site2 with a NIC ---
node2 = slice.add_node(name=node2_name, site=site2)
iface2 = node2.add_component(model='NIC_Basic', name=node2_nic_name).get_interfaces()[0]

# --- Create the L2 network connecting both interfaces ---
# Since interfaces are on different sites, this becomes a WAN L2 circuit
# No subnet specified -- we will configure IPs manually after the slice is active
net1 = slice.add_l2network(name=network_name, interfaces=[iface1, iface2])

# Submit the slice request -- blocks until ready (~3-5 min)
slice.submit();

## Step 4: Observe the Slice's Attributes

Before configuring IPs, let us inspect what FABRIC has provisioned. This is useful for understanding the topology and verifying that the WAN L2 circuit was created correctly.

In [ ]:
# Re-fetch the slice to ensure we have the latest state
slice = fablib.get_slice(name=slice_name)

# Show slice-level information (state, expiration, project)
slice.show()

# List all nodes with their details (site, state, IPs)
slice.list_nodes()

# List all networks (shows the WAN L2 circuit)
slice.list_networks()

# List all interfaces (shows NICs and their connections)
slice.list_interfaces()

## Step 5: Manually Configure IP Addresses

The WAN L2 circuit is up but the interfaces have **no IP addresses**. Some experiments use L2 networks for non-IP protocols -- if that describes your experiment, you can skip this step and start using the raw L2 link.

For IP-based experiments, we need to:
1. Pick a subnet
2. Assign IPs to each interface

### Step 5a: Pick a Subnet

Since this is a single L2 segment (same broadcast domain), both nodes share the **same subnet**.

In [ ]:
# Import IP address management library
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network

# Define a private subnet for our WAN L2 network
subnet = IPv4Network("192.168.1.0/24")

# Generate a list of usable IPs (skip .0 which is the network address)
available_ips = list(subnet)[1:]

### Step 5b: Configure Node1

Get the interface connected to the WAN L2 network and assign it an IP.

In [ ]:
# Get Node1 and its interface on the WAN L2 network
node1 = slice.get_node(name=node1_name)        
node1_iface = node1.get_interface(network_name=network_name) 

# Pop the first available IP (192.168.1.1)
node1_addr = available_ips.pop(0)

# Assign the IP address to the interface with the subnet mask
node1_iface.ip_addr_add(addr=node1_addr, subnet=subnet)

# Verify: show the interface configuration inside the VM
stdout, stderr = node1.execute(f'ip addr show {node1_iface.get_device_name()}')

### Step 5c: Configure Node2

Repeat for the second node.

In [ ]:
# Get Node2 and its interface on the WAN L2 network
node2 = slice.get_node(name=node2_name)        
node2_iface = node2.get_interface(network_name=network_name)  

# Pop the next available IP (192.168.1.2)
node2_addr = available_ips.pop(0)

# Assign the IP address to the interface
node2_iface.ip_addr_add(addr=node2_addr, subnet=subnet)

# Verify: show the interface configuration
stdout, stderr = node2.execute(f'ip addr show {node2_iface.get_device_name()}')

## Step 6: Run the Experiment

We verify cross-site L2 connectivity by pinging Node2 from Node1. Since both nodes are on the same subnet (same L2 broadcast domain), no routing is needed.

<div class="fab-warning">

**Note:** The round-trip time reflects the physical distance between the two FABRIC sites. A WAN L2 link between distant sites (e.g., east coast to west coast) will have noticeably higher latency than a local L2 network.

</div>

In [ ]:
# Ping Node2 from Node1 over the WAN L2 circuit
node1 = slice.get_node(name=node1_name)        

stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

<div class="fab-success">

**Success!** If you see ping replies, your two nodes on different FABRIC sites are communicating over a dedicated WAN Layer 2 circuit -- as if they were on the same physical switch.

</div>

## Step 7: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- WAN L2 circuits consume backbone bandwidth, so cleanup is especially important.

</div>

In [ ]:
# Delete the slice and release all resources
slice = fablib.get_slice(name=slice_name)
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `ip_addr_add()` fails | Interface not found or wrong network name | Verify `network_name` matches the name used in `add_l2network()` |
| `ping` fails between nodes | IPs not assigned or WAN circuit not established | Run `ip addr show` on both nodes; check `list_networks()` for circuit status |
| Network created as local instead of WAN | Both nodes on the same site | Ensure `site1` and `site2` are different |
| Slice stuck in `Configuring` | Site may be busy or backbone link unavailable | Try different sites |
| `No resources available` | Site lacks NIC capacity | Choose different sites or try `NIC_ConnectX_6` |
| IP address conflict | Same IP assigned to both nodes | Ensure you `pop()` different IPs from the available list |
| High latency | Sites are geographically distant | Expected -- L2 WAN latency reflects physical distance |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_random_sites(count)` | Get a list of distinct random site names | [get_random_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_sites) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `fablib.get_slice(name)` | Retrieve an existing slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `slice.add_l2network(name, interfaces)` | Add a Layer 2 network | [add_l2network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l2network) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.show()` | Display slice attributes | [show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show) |
| `slice.list_nodes()` | List all nodes in the slice | [list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodes) |
| `slice.list_networks()` | List all networks in the slice | [list_networks](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_networks) |
| `slice.list_interfaces()` | List all interfaces in the slice | [list_interfaces](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_interfaces) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |
| `node.add_component(model, name)` | Add a NIC or other component | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `node.get_interface(network_name)` | Get the interface connected to a network | [get_interface](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.get_interface) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `iface.ip_addr_add(addr, subnet)` | Assign an IP address to an interface | [ip_addr_add](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.ip_addr_add) |
| `iface.get_device_name()` | Get the Linux device name of an interface | [get_device_name](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_device_name) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **L2 WAN Auto** | [create_l2network_wide_area_auto](./create_l2network_wide_area_auto.ipynb) | Let FABlib assign IPs automatically on WAN L2 |
| **L2 Local Manual** | [create_l2network_basic_manual](../create_l2network_basic/create_l2network_basic_manual.ipynb) | Local Ethernet with manual IP configuration |
| **FABnet IPv4 Manual** | [create_l3network_fabnet_ipv4_manual](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_manual.ipynb) | Layer 3 networking with manual IP configuration |
| **FABnet IPv4 Full Auto** | [create_l3network_fabnet_ipv4_full_auto](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_full_auto.ipynb) | Simplest cross-site connectivity |
| **Hello FABRIC** | [hello_fabric](../../hello_fabric/hello_fabric.ipynb) | Start from the basics |